# Travel Planner with Checkpoints (LangGraph)

This notebook builds a workflow that:
1. Gathers traveler preferences
2. Generates itinerary options
3. Pauses for approval
4. Refines the final plan after approval


## Why Checkpoints

A planner should not auto-finalize without user review.
We checkpoint state before refinement, let the user choose or edit, then resume.


## Workflow Graph

```text
START
  -> gather_preferences
  -> generate_itinerary_options
  -> [PAUSE before refine_plan]
  -> refine_plan
  -> END
```


## Step-by-Step State Transition

1. `gather_preferences` writes a normalized preference profile.
2. `generate_itinerary_options` reads that profile and writes multiple options.
3. Pause occurs before `refine_plan` so user can approve/adjust preferences.
4. `refine_plan` reads approved notes and produces the final itinerary.


## Step 1 - Install and Import

In [1]:
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'langgraph'])
print('langgraph installed')


langgraph installed


In [2]:
from typing import List, TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph

print('Imports ready')


E:\Github\Machine-Learning-Projects\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


Imports ready


## Step 2 - Define State

In [3]:
class TravelState(TypedDict):
    raw_request: str
    preferences_summary: str
    itinerary_options: List[str]
    approval_decision: str
    approval_notes: str
    final_itinerary: str

print('State schema initialized')


State schema initialized


## Step 3 - Sample Request

In [4]:
sample_request = (
    'Plan a 4-day Tokyo trip for two adults in early October. Budget is moderate. '
    'Interested in food, neighborhoods, cultural sites, and one day trip. '
    'Prefer not to rush and want evening flexibility.'
)

print('Sample request loaded')


Sample request loaded


## Step 4 - Node 1: Gather Preferences

In [5]:
def gather_preferences(state: TravelState) -> dict:
    text = state['raw_request'].lower()

    budget = 'moderate' if 'moderate' in text else 'unspecified'
    duration = '4 days' if '4-day' in text or '4 day' in text else 'unspecified'
    pace = 'relaxed' if 'not to rush' in text or 'flexibility' in text else 'balanced'

    interests = []
    if 'food' in text:
        interests.append('food experiences')
    if 'neighborhood' in text:
        interests.append('neighborhood exploration')
    if 'cultural' in text:
        interests.append('cultural landmarks')
    if 'day trip' in text:
        interests.append('single day trip')

    summary = (
        f'Destination intent detected from request. Duration={duration}; Budget={budget}; '
        f'Pace={pace}; Interests=' + ', '.join(interests)
    )

    print('gather_preferences complete')
    return {'preferences_summary': summary}


## Step 5 - Node 2: Generate Itinerary Options

In [6]:
def generate_itinerary_options(state: TravelState) -> dict:
    options = []

    options.append(
        'Option A (City Core): Day 1 Asakusa/Ueno, Day 2 Shibuya/Harajuku, '
        'Day 3 Tsukiji + Ginza + evening izakaya, Day 4 flexible museums and shopping.'
    )

    options.append(
        'Option B (Culture + Day Trip): Day 1 Meiji Shrine/Shinjuku Gyoen, '
        'Day 2 Akihabara + Kanda food walk, Day 3 Kamakura day trip, '
        'Day 4 neighborhood cafe trail and relaxed evening.'
    )

    options.append(
        'Option C (Food-Forward): Day 1 Tokyo Station food halls + Nihonbashi, '
        'Day 2 ramen and yakitori district crawl, Day 3 Yokohama day trip, '
        'Day 4 market brunch and open block for shopping/rest.'
    )

    print('generate_itinerary_options complete with', len(options), 'options')
    return {'itinerary_options': options}


## Checkpoint Design

We compile with:
- `MemorySaver` checkpointer
- `interrupt_before=['refine_plan']`

This pauses right after options are generated so the user can approve or modify direction.


## Step 6 - Node 3: Refine Final Plan

In [7]:
def refine_plan(state: TravelState) -> dict:
    decision = state.get('approval_decision', '').lower()
    notes = state.get('approval_notes', '')

    chosen = 'Option B'
    if 'option a' in decision:
        chosen = 'Option A'
    elif 'option c' in decision:
        chosen = 'Option C'

    final_lines = []
    final_lines.append('Final Itinerary Plan')
    final_lines.append('Chosen baseline: ' + chosen)
    final_lines.append('Preferences summary: ' + state['preferences_summary'])
    final_lines.append('User approval notes: ' + notes)
    final_lines.append('')

    for option in state['itinerary_options']:
        if chosen in option:
            final_lines.append('Selected route detail:')
            final_lines.append(option)
            break

    final_lines.append('Refinement notes: keep one open evening and limit daily major stops to avoid fatigue.')

    final_text = '\n'.join(final_lines)
    print('refine_plan complete')
    return {'final_itinerary': final_text}


## Step 7 - Build Graph With Pause Point

In [8]:
builder = StateGraph(TravelState)
builder.add_node('gather_preferences', gather_preferences)
builder.add_node('generate_itinerary_options', generate_itinerary_options)
builder.add_node('refine_plan', refine_plan)

builder.add_edge(START, 'gather_preferences')
builder.add_edge('gather_preferences', 'generate_itinerary_options')
builder.add_edge('generate_itinerary_options', 'refine_plan')
builder.add_edge('refine_plan', END)

memory = MemorySaver()
app = builder.compile(checkpointer=memory, interrupt_before=['refine_plan'])

print('Graph compiled with checkpoint pause before refinement')


Graph compiled with checkpoint pause before refinement


## Step 8 - Phase 1 Run (Gather + Options, Then Pause)

In [9]:
config = {'configurable': {'thread_id': 'travel-checkpoint-37-thread-1'}}

initial_state = {
    'raw_request': sample_request,
    'preferences_summary': '',
    'itinerary_options': [],
    'approval_decision': '',
    'approval_notes': '',
    'final_itinerary': '',
}

phase1 = app.invoke(initial_state, config)

print('Workflow paused before refine_plan')
print('Preferences summary:')
print(phase1['preferences_summary'])
print('\nItinerary options:')
for i, opt in enumerate(phase1['itinerary_options'], start=1):
    print(f'{i}. {opt}')


gather_preferences complete
generate_itinerary_options complete with 3 options
Workflow paused before refine_plan
Preferences summary:
Destination intent detected from request. Duration=4 days; Budget=moderate; Pace=relaxed; Interests=food experiences, neighborhood exploration, cultural landmarks, single day trip

Itinerary options:
1. Option A (City Core): Day 1 Asakusa/Ueno, Day 2 Shibuya/Harajuku, Day 3 Tsukiji + Ginza + evening izakaya, Day 4 flexible museums and shopping.
2. Option B (Culture + Day Trip): Day 1 Meiji Shrine/Shinjuku Gyoen, Day 2 Akihabara + Kanda food walk, Day 3 Kamakura day trip, Day 4 neighborhood cafe trail and relaxed evening.
3. Option C (Food-Forward): Day 1 Tokyo Station food halls + Nihonbashi, Day 2 ramen and yakitori district crawl, Day 3 Yokohama day trip, Day 4 market brunch and open block for shopping/rest.


## Step 9 - Simulate Approval and Resume

In [10]:
approval_decision = 'Approve Option B with small edits'
approval_notes = (
    'Keep Option B, add one premium sushi dinner on Day 2, '
    'and keep Day 4 afternoon unplanned for rest.'
)

app.update_state(config, {'approval_decision': approval_decision, 'approval_notes': approval_notes})
phase2 = app.invoke(None, config)

print('Workflow resumed and finalized')
print('\n' + phase2['final_itinerary'])


refine_plan complete
Workflow resumed and finalized

Final Itinerary Plan
Chosen baseline: Option B
Preferences summary: Destination intent detected from request. Duration=4 days; Budget=moderate; Pace=relaxed; Interests=food experiences, neighborhood exploration, cultural landmarks, single day trip
User approval notes: Keep Option B, add one premium sushi dinner on Day 2, and keep Day 4 afternoon unplanned for rest.

Selected route detail:
Option B (Culture + Day Trip): Day 1 Meiji Shrine/Shinjuku Gyoen, Day 2 Akihabara + Kanda food walk, Day 3 Kamakura day trip, Day 4 neighborhood cafe trail and relaxed evening.
Refinement notes: keep one open evening and limit daily major stops to avoid fatigue.


## Step 10 - Inspect Checkpoint History

In [11]:
history = list(app.get_state_history(config))
print('Checkpoint count:', len(history))
for i, snap in enumerate(history):
    src = 'init'
    if snap.metadata and 'source' in snap.metadata:
        src = snap.metadata['source']
    keys = [k for k, v in snap.values.items() if v not in ('', [], None)]
    print(f'Checkpoint {i}: source={src}, populated_fields={keys}')


Checkpoint count: 6
Checkpoint 0: source=loop, populated_fields=['raw_request', 'preferences_summary', 'itinerary_options', 'approval_decision', 'approval_notes', 'final_itinerary']
Checkpoint 1: source=update, populated_fields=['raw_request', 'preferences_summary', 'itinerary_options', 'approval_decision', 'approval_notes']
Checkpoint 2: source=loop, populated_fields=['raw_request', 'preferences_summary', 'itinerary_options']
Checkpoint 3: source=loop, populated_fields=['raw_request', 'preferences_summary']
Checkpoint 4: source=loop, populated_fields=['raw_request']
Checkpoint 5: source=input, populated_fields=[]


## Key Takeaways

1. Checkpoints allow safe user review before committing final plan changes.
2. Pause/resume with same thread id keeps workflow context consistent.
3. This pattern scales to human-in-the-loop itinerary customization apps.
